# DeepSC Reconstruction Notebook

This notebook loads the trained DeepSC replication checkpoint and reconstructs sentences across all configured SNR values.

It supports:
- Qualitative reconstruction for a few random samples at all SNRs.
- Full reconstruction export for the complete test set at all SNRs.

In [1]:
from pathlib import Path
import random

import pandas as pd
import torch
from nltk.translate.bleu_score import sentence_bleu

import deepsc_replication as ds

In [2]:
ROOT = Path.cwd()
if not (ROOT / "deepsc_replication_config.json").exists():
    ROOT = ROOT / "btp_env"

config = ds.load_config(ROOT / "deepsc_replication_config.json")
ds.set_seed(int(config["dataset"]["seed"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

loaders, stoi, itos, pad_id, token_ids, max_seq_len = ds.load_repo_data(config, ROOT)
print(
    f"Loaded data | train={len(loaders['train'].dataset)} test={len(loaders['test'].dataset)} max_seq_len={max_seq_len}"
)

model_cfg = config["model"]
model_max_len = max(int(config["dataset"]["max_len"]), int(max_seq_len))

model = ds.DeepSCBaseline(
    vocab_size=len(stoi),
    d_model=int(model_cfg["d_model"]),
    nhead=int(model_cfg["nhead"]),
    num_enc_layers=int(model_cfg["num_enc_layers"]),
    num_dec_layers=int(model_cfg["num_dec_layers"]),
    dim_feedforward=int(model_cfg["dim_feedforward"]),
    channel_dim=int(model_cfg["channel_dim"]),
    channel_hidden=int(model_cfg["channel_hidden"]),
    channel_decoder_hidden=int(model_cfg["channel_decoder_hidden"]),
    max_len=model_max_len,
    pad_id=pad_id,
    dropout=float(model_cfg["dropout"]),
).to(device)

ckpt_path = ROOT / config["artifacts"]["checkpoint"]
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
model.eval()
print("Loaded checkpoint:", ckpt_path.name)
print("Eval SNRs:", config["channel"]["eval_snr_db"])

Using device: cuda
Loaded data | train=89983 test=9999 max_seq_len=57
Loaded checkpoint: deepsc_best.pt
Eval SNRs: [0, 3, 6, 9, 12, 15, 18]


In [3]:
special_ids = {token_ids["pad"], token_ids["start"], token_ids["end"], token_ids["unk"]}
unk_token = config["dataset"]["special_tokens"][3]
snr_values = config["channel"]["eval_snr_db"]

@torch.no_grad()
def reconstruct_one(encoded_ids, snr_db):
    input_ids = torch.tensor(encoded_ids, dtype=torch.long, device=device).unsqueeze(0)
    attention_mask = torch.ones_like(input_ids)

    noise_std = ds.snr_to_noise_std(float(snr_db))
    out_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        noise_std=noise_std,
        sos_id=token_ids["start"],
        eos_id=token_ids["end"],
    )

    ref_tokens = ds.decode_tokens(encoded_ids, itos, special_ids, token_ids["end"], unk_token)
    hyp_tokens = ds.decode_tokens(out_ids.squeeze(0).tolist(), itos, special_ids, token_ids["end"], unk_token)

    reference = " ".join(ref_tokens)
    hypothesis = " ".join(hyp_tokens) if hyp_tokens else "<empty>"

    # Optional BLEU1 at sentence level for this pair.
    if ref_tokens:
        bleu1 = sentence_bleu([ref_tokens], hyp_tokens if hyp_tokens else ["the"], weights=(1.0, 0, 0, 0))
    else:
        bleu1 = 0.0

    return reference, hypothesis, float(bleu1)

In [4]:
# Few-sample qualitative check across all SNRs
NUM_SAMPLES = 5

test_encoded = loaders["test"].dataset.encoded_samples
sample_indices = random.sample(range(len(test_encoded)), k=min(NUM_SAMPLES, len(test_encoded)))

rows = []
for idx in sample_indices:
    encoded_ids = test_encoded[idx]
    for snr in snr_values:
        reference, hypothesis, bleu1 = reconstruct_one(encoded_ids, snr)
        rows.append({
            "sample_idx": int(idx),
            "snr_db": float(snr),
            "reference": reference,
            "hypothesis": hypothesis,
            "bleu1": bleu1,
        })

few_df = pd.DataFrame(rows).sort_values(["sample_idx", "snr_db"]).reset_index(drop=True)
few_df

d:\btp_reso\btp_env\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(
d:\btp_reso\btp_env\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
d:\btp_reso\btp_env\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  w

,sample_idx,snr_db,reference,hypothesis,bleu1
0,409,0.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have a at report of it",0.833333
1,409,3.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have made a note of it",1.000000
2,409,6.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have made a note of it",1.000000
3,409,9.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have made a note of it",1.000000
4,409,12.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have made a note of it",1.000000
5,409,15.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have made a note of it",1.000000
6,409,18.0,"thank you , mr falconer i have made a note of it","thank you , mr falconer i have made a note of it",1.000000
7,1824,0.0,question no 60 by john mccartin h 0620 98,mr m 60 by john kratsa h a4 98,0.555556
8,1824,3.0,question no 60 by john mccartin h 0620 98,question no 60 by john mcmahon president item,0.551561
9,1824,6.0,question no 60 by john mccartin h 0620 98,question no 60 by john mcmahon h sjostedt 98,0.777778


In [5]:
for idx in sorted(few_df["sample_idx"].unique().tolist()):
    sample_df = few_df[few_df["sample_idx"] == idx].sort_values("snr_db")
    print("=" * 120)
    print("Sample:", idx)
    print("Reference:", sample_df["reference"].iloc[0])
    for _, row in sample_df.iterrows():
        print(f"SNR {int(row['snr_db']):>2} dB | BLEU1={row['bleu1']:.4f} | {row['hypothesis']}")

Sample: 409
Reference: thank you , mr falconer i have made a note of it
SNR  0 dB | BLEU1=0.8333 | thank you , mr falconer i have a at report of it
SNR  3 dB | BLEU1=1.0000 | thank you , mr falconer i have made a note of it
SNR  6 dB | BLEU1=1.0000 | thank you , mr falconer i have made a note of it
SNR  9 dB | BLEU1=1.0000 | thank you , mr falconer i have made a note of it
SNR 12 dB | BLEU1=1.0000 | thank you , mr falconer i have made a note of it
SNR 15 dB | BLEU1=1.0000 | thank you , mr falconer i have made a note of it
SNR 18 dB | BLEU1=1.0000 | thank you , mr falconer i have made a note of it
Sample: 1824
Reference: question no 60 by john mccartin h 0620 98
SNR  0 dB | BLEU1=0.5556 | mr m 60 by john kratsa h a4 98
SNR  3 dB | BLEU1=0.5516 | question no 60 by john mcmahon president item
SNR  6 dB | BLEU1=0.7778 | question no 60 by john mcmahon h sjostedt 98
SNR  9 dB | BLEU1=0.7778 | question no 60 by john mcmahon h resumed 98
SNR 12 dB | BLEU1=0.7778 | question no 60 by john mcmaho

In [7]:
# Sampled reconstruction across all SNRs (not full test split).
# Pick a value in [15, 20] as requested.
NUM_RECON_SAMPLES = 20
RANDOM_SAMPLE = True

total_test = len(test_encoded)
num_take = min(int(NUM_RECON_SAMPLES), total_test)

if RANDOM_SAMPLE:
    selected_indices = random.sample(range(total_test), k=num_take)
else:
    selected_indices = list(range(num_take))

rows = []
for i, idx in enumerate(selected_indices, start=1):
    encoded_ids = test_encoded[idx]
    for snr in snr_values:
        reference, hypothesis, bleu1 = reconstruct_one(encoded_ids, snr)
        rows.append({
            "sample_idx": int(idx),
            "snr_db": float(snr),
            "reference": reference,
            "hypothesis": hypothesis,
            "bleu1": bleu1,
        })

    if i % 5 == 0 or i == num_take:
        print(f"Processed {i}/{num_take} selected samples")

full_df = pd.DataFrame(rows).sort_values(["sample_idx", "snr_db"]).reset_index(drop=True)
print("Selected samples:", num_take)
print("Rows:", len(full_df))
full_df.head()

Processed 5/20 selected samples
Processed 10/20 selected samples
Processed 15/20 selected samples
Processed 20/20 selected samples
Selected samples: 20
Rows: 140


,sample_idx,snr_db,reference,hypothesis,bleu1
0,434,0.0,"mr martinez , you are an admirable jurist you ...","you no , and see an president not at the end o...",0.470779
1,434,3.0,"mr martinez , you are an admirable jurist you ...","mr martinez the you , an mrs , you know that t...",0.777778
2,434,6.0,"mr martinez , you are an admirable jurist you ...","mr martinez , you are an begin commission you ...",0.925926
3,434,9.0,"mr martinez , you are an admirable jurist you ...","mr martinez , you are an gentlemen , the know ...",0.888889
4,434,12.0,"mr martinez , you are an admirable jurist you ...","mr martinez , you are an gentlemen , you know ...",0.925926


In [8]:
# Save sampled reconstruction outputs
out_csv = ROOT / "deepsc_reconstruction_samples_all_snrs.csv"
out_parquet = ROOT / "deepsc_reconstruction_samples_all_snrs.parquet"

full_df.to_csv(out_csv, index=False)
full_df.to_parquet(out_parquet, index=False)

print("Saved:", out_csv)
print("Saved:", out_parquet)

summary = full_df.groupby("snr_db", as_index=False)["bleu1"].mean().rename(columns={"bleu1": "avg_sentence_bleu1"})
summary

Saved: d:\btp_reso\btp_env\deepsc_reconstruction_samples_all_snrs.csv
Saved: d:\btp_reso\btp_env\deepsc_reconstruction_samples_all_snrs.parquet


,snr_db,avg_sentence_bleu1
0,0.0,0.454226
1,3.0,0.657419
2,6.0,0.814022
3,9.0,0.894182
4,12.0,0.914351
5,15.0,0.912693
6,18.0,0.925462


In [9]:
# Print all reconstructed samples (all selected indices x all SNRs)
if full_df.empty:
    print("No reconstructions found. Run the reconstruction cell first.")
else:
    for sample_idx in sorted(full_df["sample_idx"].unique().tolist()):
        sample_df = full_df[full_df["sample_idx"] == sample_idx].sort_values("snr_db")
        print("=" * 120)
        print("Sample:", sample_idx)
        print("Reference:", sample_df["reference"].iloc[0])
        for _, row in sample_df.iterrows():
            print(f"SNR {int(row['snr_db']):>2} dB | BLEU1={row['bleu1']:.4f} | {row['hypothesis']}")

Sample: 434
Reference: mr martinez , you are an admirable jurist you know that this is not the right time for explanations of vote , which are at the end
SNR  0 dB | BLEU1=0.4708 | you no , and see an president not at the end of this in it mr commissioner thank vote of the which
SNR  3 dB | BLEU1=0.7778 | mr martinez the you , an mrs , you know that this is not the right for before explanations of vote , which is the joint end
SNR  6 dB | BLEU1=0.9259 | mr martinez , you are an begin commission you know that this is not the right time for explanations of vote , which are at the end
SNR  9 dB | BLEU1=0.8889 | mr martinez , you are an gentlemen , the know that this is not the right time for explanations of vote , which are at the end
SNR 12 dB | BLEU1=0.9259 | mr martinez , you are an gentlemen , you know that this is not the right time for explanations of vote , which are at the end
SNR 15 dB | BLEU1=0.9259 | mr martinez , you are an gentlemen , you know that this is not the right time 